# Project: Labels vs. Scale: Breaking Even on Banking77 with Encoders and Decoders
#### Dataset: banking77 

### Dataset splits

This notebook creates every training set the experiments will use, so the encoder and the decoder are always fine-tuned on **identical examples**.

**Design**

| Piece | What it is | Why |
|---|---|---|
| Validation set | `VAL_PER_CLASS` examples per intent, taken out of train once (fixed seed) | Same validation set for every run; balanced like the test set |
| Training pool | train minus validation | Everything the subsets are drawn from |
| Balanced Subsets | `k` examples per intent for each `k` in `K_VALUES` and each seed | The data-size grid; **nested** within a seed (k=1 is inside k=3, and so on) |
| Proportional subset | about `PROP_FRACTION` of the pool, per seed. | It contains the largest balanced subset. |
| Full set | the whole training pool | Fine tuning with all training data |
| Test set | unchanged `data/raw/test.parquet` | Touched only for final evaluation |

All files saved to `data/splits/`. Every row keeps its `id` (its row position in `data/raw/train.parquet`), so any subset can be traced back to the raw data.

#### Imports

In [44]:
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Find the project root 
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "scripts" / "download_data.py").exists()), cwd)
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "splits"
OUT.mkdir(parents=True, exist_ok=True)


VAL_PER_CLASS = 5              # validation examples taken from every intent
VAL_SEED = 42                  # seed for the validation draw (fixed, so every run shares one validation set)
SEEDS = [0, 1, 2]              # one set of nested subsets per seed
K_VALUES = [1, 3, 5, 10, 20, 30]   # examples per intent; must be <= (smallest intent size - VAL_PER_CLASS)
PROP_FRACTION = 0.5            # Share of the training pool used for the proportional subset

#### Load the raw data

In [45]:
train = pd.read_parquet(RAW / "train.parquet").reset_index(drop=True)
test = pd.read_parquet(RAW / "test.parquet").reset_index(drop=True)
label_names = json.load(open(RAW / "label_names.json"))
n_classes = len(label_names)

# Stable row id = position in the raw train file. 
train["id"] = train.index

print("Train:", train.shape, "| Test:", test.shape, "| Intents:", n_classes)

#print(train.head(2))

Train: (10003, 3) | Test: (3080, 2) | Intents: 77


In [46]:
class_sizes = train.label.value_counts()
max_balanced_k = class_sizes.min() - VAL_PER_CLASS

print("Smallest intent in train:", class_sizes.min())
print("Largest balanced k available after the validation split:", max_balanced_k)

assert max(K_VALUES) <= max_balanced_k, (
    f"K_VALUES goes up to {max(K_VALUES)} but only {max_balanced_k} examples per intent remain "
    f"after holding out {VAL_PER_CLASS} for validation. Lower K_VALUES or VAL_PER_CLASS."
)


Smallest intent in train: 35
Largest balanced k available after the validation split: 30


#### Validation split
The same number of examples is drawn from every intent (a balanced validation set, like the balanced test set). The draw happens **once**, with a fixed seed, so all runs, seeds and both models share one validation set. The seeds only change which training examples are chosen.

In [47]:

val = train.groupby("label", group_keys=False).sample(n=VAL_PER_CLASS, random_state=VAL_SEED)

# The training pool is everything else (the index is the id, so dropping by index drops by id)
training_pool = train.drop(index=val.index)

print("Validation:", len(val), "| Training pool:", len(training_pool))
print("Training pool examples per intent: min", training_pool.label.value_counts().min(), "| max", training_pool.label.value_counts().max())

Validation: 385 | Training pool: 9618
Training pool examples per intent: min 30 | max 182


#### Nested subsets
For each seed, every intent's training pool examples are put in one random order, **once**. The subset for a given `k` takes the first `k` examples of each intent's order. Because every `k` reads from the same ordering, a smaller subset is automatically contained in every larger one (k=1 inside k=3 inside k=5 ...).

Proportional subset- Beyond the largest balanced size, intents cannot stay balanced, because the smallest intent runs out of examples. The proportional subset keeps the whole largest balanced subset and adds a share of each intent's remaining examples, so the total is about `PROP_FRACTION` of the training pool. It is nested: the largest balanced subset sits inside it, and it sits inside the full training set.

A different seed gives a different ordering, so the three seeds are three independent draws of the labeled data.

In [48]:
# Training Pool ids grouped by intent, computed once: {label id: array of ids}
ids_by_class = {label: g["id"].to_numpy() for label, g in training_pool.groupby("label")}

# The largest balanced size. The proportional subset keeps all of it and adds more
K_MAX = max(K_VALUES)

# Share of each intent's remaining examples (beyond K_MAX) that brings the total to PROP_FRACTION of the pool
extra_share = (PROP_FRACTION * len(training_pool) - K_MAX * n_classes) / (len(training_pool) - K_MAX * n_classes)
assert 0 < extra_share < 1, "PROP_FRACTION must be larger than the K_MAX subset and smaller than the full pool."

subsets = {}        # (seed, k) -> DataFrame, balanced subsets
prop_subsets = {}   # NEW: seed -> DataFrame, proportional subset

for seed in SEEDS:
    rng = np.random.default_rng(seed)

    # One random ordering of each intent's training pool ids for this seed (intents visited in a fixed order)
    ordered = {label: rng.permutation(ids_by_class[label]) for label in range(n_classes)}

    for k in K_VALUES:
        # First k ids of every intent -> k * 77 examples, exactly k per intent
        chosen = np.concatenate([ordered[label][:k] for label in range(n_classes)])
        subsets[(seed, k)] = training_pool.loc[chosen].sort_values("id")
    
    # Proportional subset. For each intent take its first K_MAX ids (the balanced subset)
    # plus extra_share of the ids that remain.
    take = [K_MAX + round(extra_share * (len(ordered[label]) - K_MAX)) for label in range(n_classes)]
    chosen = np.concatenate([ordered[label][:n] for label, n in enumerate(take)])
    prop_subsets[seed] = training_pool.loc[chosen].sort_values("id")

print("Created", len(subsets), "balanced subsets =", len(SEEDS), "seeds x", len(K_VALUES), "sizes")
print("Created", len(prop_subsets), "proportional subsets with", len(prop_subsets[SEEDS[0]]), "examples each")

Created 18 balanced subsets = 3 seeds x 6 sizes
Created 3 proportional subsets with 4807 examples each


#### Save

In [49]:
# Columns kept in every saved file
cols = ["id", "text", "label"]

# Validation set and the full training pool, sorted by id so the files are easy to compare
val[cols].sort_values("id").to_parquet(OUT / "val.parquet", index=False)
training_pool[cols].sort_values("id").to_parquet(OUT / "full_train.parquet", index=False)

# One file per (seed, k) subset
for (seed, k), df in subsets.items():
    df[cols].to_parquet(OUT / f"train_k{k}_seed{seed}.parquet", index=False)

# One file per seed for the proportional subset, e.g. train_p50_seed0.parquet
PROP_TAG = round(PROP_FRACTION * 100)
for seed, df in prop_subsets.items():
    df[cols].to_parquet(OUT / f"train_p{PROP_TAG}_seed{seed}.parquet", index=False)

manifest = {
    "val_per_class": VAL_PER_CLASS,
    "val_seed": VAL_SEED,
    "seeds": SEEDS,
    "k_values": K_VALUES,
    "prop_fraction": PROP_FRACTION,
    "n_prop_subset": len(prop_subsets[SEEDS[0]]),
    "n_intents": n_classes,
    "n_val": len(val),
    "n_full_train": len(training_pool),
    "subset_sizes": {str(k): k * n_classes for k in K_VALUES},
    
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("files written to", OUT)
print(sorted(p.name for p in OUT.iterdir())[:6], "...")

files written to /Users/chandu/Desktop/Term3-Fall2026/266-NLP/Final Project/DATASCI266-Banking77-Encoder-vs-Decoder/data/splits
['full_train.parquet', 'manifest.json', 'train_k10_seed0.parquet', 'train_k10_seed1.parquet', 'train_k10_seed2.parquet', 'train_k1_seed0.parquet'] ...


#### Sanity checks
The checks **reload the saved files from disk**, so they verify what the experiments will actually read. The cell raises an error at the end if any check fails.

**Validation set and full training set**
- Validation has exactly `VAL_PER_CLASS` examples for every intent.
- Validation and the full training set share no examples (by id).
- Validation plus the full training set equals all raw train rows.
- No text appears in both validation and the full training set (lowercased, trimmed).

**Saved data vs. raw data**
- Saved text and label match the raw train file for every id.

**Training subsets**
- Each subset has exactly `k` examples for every intent.
- No subset contains a repeated example.
- Every subset comes from the training pool only.
- No subset contains a validation example.

**Nesting and seeds**
- Subsets are nested within each seed (k=1 inside k=3 inside k=5 ...).
- Different seeds give different subsets at the smallest `k`.

**Proportional subset**
- It has no repeated example.
- It comes from the training pool only and has no validation example.
- It contains the largest balanced subset of the same seed.
- Every intent has at least `K_MAX` examples and no more than the pool holds.
- Its size is within 1 percentage point of `PROP_FRACTION` of the pool.

**Test set**
- Test set is unchanged: 3,080 rows, 40 per intent.

In [50]:
# Reload everything that was saved
val_d = pd.read_parquet(OUT / "val.parquet")
full_d = pd.read_parquet(OUT / "full_train.parquet")
subs_d = {(s, k): pd.read_parquet(OUT / f"train_k{k}_seed{s}.parquet") for s in SEEDS for k in K_VALUES}
PROP_TAG = round(PROP_FRACTION * 100)
prop_d = {s: pd.read_parquet(OUT / f"train_p{PROP_TAG}_seed{s}.parquet") for s in SEEDS}

failures = []
def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    if not ok:
        failures.append(name)

val_ids, full_ids = set(val_d.id), set(full_d.id)

# ---- Validation and full training set ----
check("validation has exactly VAL_PER_CLASS examples for every intent",
      val_d.label.nunique() == n_classes and (val_d.label.value_counts() == VAL_PER_CLASS).all())
check("validation and full training set share no examples (by id)", val_ids.isdisjoint(full_ids))
check("validation + full training set = all raw train rows",
      len(val_d) + len(full_d) == len(train) and (val_ids | full_ids) == set(train.id))

# Case-insensitive text overlap between validation and training (catches duplicate texts under different ids)
norm = lambda s: s.str.lower().str.strip()
check("no text appears in both validation and full training set",
      set(norm(val_d.text)).isdisjoint(set(norm(full_d.text))))

# Saved rows must equal the raw rows with the same id
raw_by_id = train.set_index("id")
same = all((raw_by_id.loc[d.id, "text"].to_numpy() == d.text.to_numpy()).all()
           and (raw_by_id.loc[d.id, "label"].to_numpy() == d.label.to_numpy()).all()
           for d in [val_d, full_d, *subs_d.values(), *prop_d.values()])
check("saved text and label match the raw train file for every id", same)

# ---- Every subset ----
check("every subset has exactly k examples for every intent (k x 77 rows in total)",
      all(len(d) == k * n_classes and d.label.nunique() == n_classes and (d.label.value_counts() == k).all()
          for (s, k), d in subs_d.items()))
check("no subset contains a repeated example", all(d.id.is_unique for d in subs_d.values()))
check("every subset is drawn from the training pool only",
      all(set(d.id) <= full_ids for d in subs_d.values()))
check("no subset contains a validation example", all(set(d.id).isdisjoint(val_ids) for d in subs_d.values()))

# ---- Nesting within a seed ----
ks = sorted(K_VALUES)
check("subsets are nested within each seed (k=1 inside k=3 inside ...)",
      all(set(subs_d[(s, a)].id) <= set(subs_d[(s, b)].id) for s in SEEDS for a, b in zip(ks, ks[1:])))

# ---- Proportional subset ----
full_counts = full_d.label.value_counts()

def intent_counts_ok(d):
    # Every intent is present, with at least K_MAX examples and no more than the pool holds
    counts = d.label.value_counts()
    return (d.label.nunique() == n_classes
            and (counts >= K_MAX).all()
            and (counts <= full_counts.reindex(counts.index)).all())

check("proportional subset has no repeated example", all(d.id.is_unique for d in prop_d.values()))
check("proportional subset is drawn from the training pool and has no validation example",
      all(set(d.id) <= full_ids and set(d.id).isdisjoint(val_ids) for d in prop_d.values()))
check("proportional subset contains the largest balanced subset of the same seed (nested)",
      all(set(subs_d[(s, K_MAX)].id) <= set(prop_d[s].id) for s in SEEDS))
check("proportional subset has at least K_MAX and at most all pool examples for every intent",
      all(intent_counts_ok(d) for d in prop_d.values()))
check("proportional subset is about PROP_FRACTION of the training pool (within 1 percentage point)",
      all(abs(len(d) / len(full_d) - PROP_FRACTION) < 0.01 for d in prop_d.values()))

# ---- Seeds really differ (smallest k) ----
first = [set(subs_d[(s, ks[0])].id) for s in SEEDS]
check(f"different seeds give different subsets at k={ks[0]}",
      all(first[i] != first[j] for i in range(len(first)) for j in range(i + 1, len(first))))

# ---- Test set ----
check("test set is unchanged: 40 examples for each of the 77 intents",
      len(test) == 3080 and (test.label.value_counts() == 40).all())

assert not failures, f"{len(failures)} check(s) failed: {failures}"
print("\nAll checks passed.")

PASS  validation has exactly VAL_PER_CLASS examples for every intent
PASS  validation and full training set share no examples (by id)
PASS  validation + full training set = all raw train rows
PASS  no text appears in both validation and full training set
PASS  saved text and label match the raw train file for every id
PASS  every subset has exactly k examples for every intent (k x 77 rows in total)
PASS  no subset contains a repeated example
PASS  every subset is drawn from the training pool only
PASS  no subset contains a validation example
PASS  subsets are nested within each seed (k=1 inside k=3 inside ...)
PASS  proportional subset has no repeated example
PASS  proportional subset is drawn from the training pool and has no validation example
PASS  proportional subset contains the largest balanced subset of the same seed (nested)
PASS  proportional subset has at least K_MAX and at most all pool examples for every intent
PASS  proportional subset is about PROP_FRACTION of the trainin

#### Summary
The three seeds draw from the same finite training pool, so their subsets overlap more as k grows. Variation between seeds is therefore smaller at large k.

In [51]:
from itertools import combinations

rows = []
for k in ks:
    sets = [set(subs_d[(s, k)].id) for s in SEEDS]
    # Average share of examples two seeds have in common (1.0 = identical subsets)
    overlap = np.mean([len(a & b) / len(a) for a, b in combinations(sets, 2)])
    rows.append({"k per intent": k, "total examples": k * n_classes, "avg overlap between seeds": round(overlap, 3)})

# Proportional subset, same overlap calculation using each seed's proportional subset
prop_sets = [set(prop_d[s].id) for s in SEEDS]
prop_overlap = np.mean([len(a & b) / len(a) for a, b in combinations(prop_sets, 2)])
rows.append({"k per intent": f"{PROP_TAG}% of pool", "total examples": len(prop_d[SEEDS[0]]),
             "avg overlap between seeds": round(prop_overlap, 3)})

rows.append({"k per intent": "full", "total examples": len(full_d), "avg overlap between seeds": 1.0})
pd.DataFrame(rows)

,k per intent,total examples,avg overlap between seeds
0,1,77,0.009
1,3,231,0.026
2,5,385,0.048
3,10,770,0.087
4,20,1540,0.181
5,30,2310,0.268
6,50% of pool,4807,0.509
7,full,9618,1.000


#### Cross-check with the EDA
The EDA found a few queries that appear in both train and test. The  test set is not edited, and are only counted here so the numbers can be reported.

In [52]:
overlap_test = set(norm(full_d.text)) & set(norm(test.text))
print("Queries in the training pool that also appear in test:", len(overlap_test))
print("EDA found 6 across all of train; fewer here means some of them were moved to validation")

Queries in the training pool that also appear in test: 5
EDA found 6 across all of train; fewer here means some of them were moved to validation
